# Description

In this notebook, I will explore the benchmark Human Eval and using GPT4 to generate it.

In [1]:
import os 
import sys
import time
import numpy as np 
import pandas as pd 
import re
import io
import re
import ast
import types
import unittest
import importlib
from typing import List, Tuple, Dict, Any, Set
import load_dotenv
from openai import OpenAI

# 1. Load data

In [2]:
PATH_CSV_DATA = "data/raw_data/human_eval.csv"

In [3]:
load_dotenv.load_dotenv()

# OPEN_AI_API = os.getenv("OPEN_AI_API")
OPEN_AI_API = os.getenv("OPEN_AI_API_v2")
if OPEN_AI_API is None:
    raise ValueError("OPEN_AI_API environment variable not set")
else:
    print("API key loaded successfully")

client = OpenAI(api_key=OPEN_AI_API)

API key loaded successfully


In [4]:
df = pd.read_csv(PATH_CSV_DATA)
print(f"Dataframe shape: {df.shape}")
df.sample(1)

Dataframe shape: (164, 5)


,task_id,prompt,canonical_solution,test,entry_point
140,HumanEval/140,"\ndef fix_spaces(text):\n """"""\n Given a ...","new_text = """"\n i = 0\n start, end =...",def check(candidate):\n\n # Check some simp...,fix_spaces


In [5]:
idx = np.random.randint(0, df.shape[0])

code_description = df.loc[idx, "prompt"]
test_case = df.loc[idx, "test"]
entry_point = df.loc[idx, "entry_point"]

print("Code description:")
print(code_description)
print("=" * 20)
print("Test Case:")
print(test_case)

Code description:

def hex_key(num):
    """You have been tasked to write a function that receives 
    a hexadecimal number as a string and counts the number of hexadecimal 
    digits that are primes (prime number, or a prime, is a natural number 
    greater than 1 that is not a product of two smaller natural numbers).
    Hexadecimal digits are 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, A, B, C, D, E, F.
    Prime numbers are 2, 3, 5, 7, 11, 13, 17,...
    So you have to determine a number of the following digits: 2, 3, 5, 7, 
    B (=decimal 11), D (=decimal 13).
    Note: you may assume the input is always correct or empty string, 
    and symbols A,B,C,D,E,F are always uppercase.
    Examples:
    For num = "AB" the output should be 1.
    For num = "1077E" the output should be 2.
    For num = "ABED1A33" the output should be 4.
    For num = "123456789ABCDEF0" the output should be 6.
    For num = "2020" the output should be 2.
    """

Test Case:
def check(candidate):

    # Check some sim

# 2. Using GPT4 to generate sample

## 2.1. Generate code

In [6]:
def extract_function(llm_text):
    # 1) Grab text between <code>...</code>
    m = re.search(r"<code>\s*(.*?)\s*</code>", llm_text, flags=re.S|re.M)
    if not m:
        raise ValueError("No <code> block found")
    code = m.group(1)

    # 2) Optionally, if the model sometimes adds backticks, strip them
    code = re.sub(r"^```(?:python)?\s*|\s*```$", "", code.strip())

    return code

In [7]:
constraints = """
Output only a complete and valid Python code for this function. Do not change the provided function signature.
Do NOT print any markdown or include the test cases in your output.
Wrap your output strictly between the markers:
<code>
... your code ...
</code>
"""

input_prompt = f"""write a complete python function
based on the following description:\n{code_description}.\n
with the following constraints:\n{constraints}
"""

print("Input prompt to:")
print(input_prompt)

Input prompt to:
write a complete python function
based on the following description:

def hex_key(num):
    """You have been tasked to write a function that receives 
    a hexadecimal number as a string and counts the number of hexadecimal 
    digits that are primes (prime number, or a prime, is a natural number 
    greater than 1 that is not a product of two smaller natural numbers).
    Hexadecimal digits are 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, A, B, C, D, E, F.
    Prime numbers are 2, 3, 5, 7, 11, 13, 17,...
    So you have to determine a number of the following digits: 2, 3, 5, 7, 
    B (=decimal 11), D (=decimal 13).
    Note: you may assume the input is always correct or empty string, 
    and symbols A,B,C,D,E,F are always uppercase.
    Examples:
    For num = "AB" the output should be 1.
    For num = "1077E" the output should be 2.
    For num = "ABED1A33" the output should be 4.
    For num = "123456789ABCDEF0" the output should be 6.
    For num = "2020" the output should b

In [8]:
response = client.chat.completions.create(
    model="gpt-4o",  # or "gpt-4o" 
    messages=[
        {"role": "system", "content": "You are an expert in Python."},
        {"role": "user", "content": input_prompt},
    ],
)

output = response.choices[0].message.content

print("Response from OpenAI:")
print(output)

Response from OpenAI:
<code>
def hex_key(num):
    """You have been tasked to write a function that receives 
    a hexadecimal number as a string and counts the number of hexadecimal 
    digits that are primes."""
    
    # Hexadecimal prime digits
    prime_hex_digits = {'2', '3', '5', '7', 'B', 'D'}
    
    count = 0
    
    # Check each character in the string
    for char in num:
        if char in prime_hex_digits:
            count += 1
    
    return count
</code>


We can extract the complete code

In [9]:
completed_code = extract_function(output)
print(f"The complete code:\n")
print(completed_code)

The complete code:

def hex_key(num):
    """You have been tasked to write a function that receives 
    a hexadecimal number as a string and counts the number of hexadecimal 
    digits that are primes."""
    
    # Hexadecimal prime digits
    prime_hex_digits = {'2', '3', '5', '7', 'B', 'D'}
    
    count = 0
    
    # Check each character in the string
    for char in num:
        if char in prime_hex_digits:
            count += 1
    
    return count


## 2.2. Evaluate the generated code

In [10]:
import builtins
import typing

def create_namespace():
    ns = {}

    # 1. Standard builtins (print, len, etc.)
    ns.update({k: getattr(builtins, k) for k in dir(builtins)})

    # 2. Install common typing names (List, Optional, etc.)
    for name in typing.__all__:
        ns[name] = getattr(typing, name)

    # 3. (Optional) Add math, random, itertools, etc.
    import math, random, itertools, statistics
    ns.update({
        'math': math,
        'random': random,
        'itertools': itertools,
        'statistics': statistics,
    })

    return ns

In [11]:
def evaluate_asserts(generated_code: str, test_code: str, entry_point: str):
    # ns = {}
    ns = create_namespace()
    
    # 1. Exec both code strings
    exec(generated_code, ns)
    exec(test_code, ns)

    candidate = ns[entry_point]     # the model's function
    check_fn = ns["check"]          # original check() function
    
    # 2. Parse the test code AST
    tree = ast.parse(test_code)

    # 3. Find the check() function body
    check_body = None
    for node in tree.body:
        if isinstance(node, ast.FunctionDef) and node.name == "check":
            check_body = node.body
            break

    if check_body is None:
        raise ValueError("check() function not found.")
    
    # 4. Evaluate each assert individually
    results = []
    for idx, stmt in enumerate(check_body):
        if isinstance(stmt, ast.Assert):
            # Convert AST back to executable code
            code = compile(ast.Module([stmt], type_ignores=[]), "<assert>", "exec")
            try:
                exec(code, {**ns, "candidate": candidate})
                results.append(("pass", None))
            except Exception as e:
                results.append(("fail", repr(e)))

    # 5. Compute pass percentage
    total = len(results)
    passed = sum(1 for r, _ in results if r == "pass")
    percentage = passed / total if total > 0 else 0.0

    return {
        "total_asserts": total,
        "passed": passed,
        "percentage": percentage,
        "detail": results
    }

In [12]:
result = evaluate_asserts(completed_code, test_case, entry_point)
print(result)

{'total_asserts': 7, 'passed': 7, 'percentage': 1.0, 'detail': [('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None)]}


# 3. Run through all sample

In [13]:
list_df = []

list_num_out_token = []

start_time = time.time()

for idx in range(df.shape[0]):
    if idx % 10 == 0:
        print(f"Processing idx={idx}/{df.shape[0]}...")
    
    try:
        # 1. Prepare input prompt
        code_description = df.loc[idx, "prompt"]
        test_case = df.loc[idx, "test"]
        entry_point = df.loc[idx, "entry_point"]

        input_prompt = f"""write a complete python function
        based on the following description:\n{code_description}.\n
        with the following constraints:\n{constraints}
        """

        # 2. Generate code with GPT-4o
        response = client.chat.completions.create(
            model="gpt-4o",  # or "gpt-4o" 
            messages=[
                {"role": "system", "content": "You are an expert in Python."},
                {"role": "user", "content": input_prompt},
            ],
        )

        output = response.choices[0].message.content
        completed_code = extract_function(output)
        num_out_token = response.usage.completion_tokens
        list_num_out_token.append(num_out_token)
        
        
        # 3. Evaluate the generated code
        result = evaluate_asserts(completed_code, test_case, entry_point)
        total_asserts = result["total_asserts"]
        passed_asserts = result["passed"]
        percentage = result["percentage"]
        
        list_df.append({
            "description": code_description,
            "generated_code": completed_code,
            "test_case": test_case,
            "entry_point": entry_point,
            "total_asserts": total_asserts,
            "passed_asserts": passed_asserts,
            "percentage": percentage,
        })
    except Exception as e:
        print(f"Error at idx={idx}: {e}")
        continue
    
end_time = time.time()
avg_time_per_example = (end_time - start_time) / len(list_df)
print(f"Average time per example: {avg_time_per_example:.2f} seconds")

avg_output_tokens = sum(list_num_out_token) / len(list_num_out_token)
print(f"Average number of output tokens: {avg_output_tokens:.2f}")

Processing idx=0/164...
Processing idx=10/164...
Processing idx=20/164...
Processing idx=30/164...
Processing idx=40/164...
Processing idx=50/164...
Processing idx=60/164...
Processing idx=70/164...
Processing idx=80/164...
Processing idx=90/164...
Processing idx=100/164...
Processing idx=110/164...
Processing idx=120/164...
Processing idx=130/164...
Processing idx=140/164...
Processing idx=150/164...
Processing idx=160/164...
Average time per example: 1.51 seconds
Average number of output tokens: 123.95


In [14]:
output_df = pd.DataFrame(list_df)
print(f'Output dataframe shape: {output_df.shape}')
output_df.sample()

Output dataframe shape: (164, 7)


,description,generated_code,test_case,entry_point,total_asserts,passed_asserts,percentage
138,"\ndef is_equal_to_sum_even(n):\n """"""Evaluat...","def is_equal_to_sum_even(n):\n """"""Evaluate ...",def check(candidate):\n assert candidate(4)...,is_equal_to_sum_even,8,8,1.0


In [15]:
# # Save to CSV
# output_df.to_csv("data/human_eval_generated_gpt4o.csv", index=False)

## 3.1. Check generated code

In [16]:
average_percentage = output_df["percentage"].mean()
print(f"Average pass percentage over all samples: {average_percentage:.2%}")    

Average pass percentage over all samples: 94.75%
